# SNI-21 VA-DCP — pilot 200 + validation screening

Notebook ini menjalankan workflow terkunci berikut:

1. membuat/reuse A0 serta 200 scene sintetis A1 dan A2;
2. menampilkan audit visual sebelum training;
3. melatih A0/A1/A2 selama 10 epoch, seed 42;
4. mengevaluasi **validation nyata saja**;
5. menyimpan checkpoint ke Google Drive dan mencadangkannya ke repo Hugging Face privat.

**Test tidak dibuka.** Pilih runtime **T4 GPU sebelum menjalankan cell pertama**. Pembuatan scene pilot bersifat CPU-bound dan dapat berlangsung sekitar satu jam; progress akan tercetak.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if not (REPO / '.git').is_dir():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
assert torch.cuda.is_available(), 'GPU tidak aktif. Runtime > Change runtime type > T4 GPU, lalu ulangi dari awal.'
from coffee_detector.run_sni21_colab_setup import run_sni21_colab_setup
from coffee_detector.run_vadcp_ablation import run_vadcp_ablation
print('REPO DAN GPU SIAP:', torch.cuda.get_device_name(0))
print('TRAINING BELUM DIMULAI; TEST TETAP TERKUNCI.')

In [ ]:
from getpass import getpass
from google.colab import userdata
from huggingface_hub import HfApi

DRIVE = Path('/content/drive/MyDrive')
ADRIAN_ARCHIVE = DRIVE / 'coffee-sni-detection-fullscene-v1/adrian_detection.tar'
FARUQ_ARCHIVE = DRIVE / 'coffee-sni-detection-fullscene-v1/faruq_segmentation.tar'
CROP_ROOT = DRIVE / 'coffee-sni-instance-crop-v1'
RESULT_ROOT = DRIVE / 'coffee-bean-detection/sni21-vadcp-pilot-results'
EVIDENCE_ROOT = DRIVE / 'coffee-bean-detection/sni21-vadcp-pilot-evidence'

assert ADRIAN_ARCHIVE.is_file(), ADRIAN_ARCHIVE
assert FARUQ_ARCHIVE.is_file(), FARUQ_ARCHIVE
assert (CROP_ROOT / 'manifest.csv').is_file(), CROP_ROOT
assert (CROP_ROOT / 'shards').is_dir(), CROP_ROOT / 'shards'

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass('Masukkan token Hugging Face write: ')
assert HF_TOKEN, 'HF_TOKEN wajib agar checkpoint tidak hanya berada di runtime.'
os.environ['HF_TOKEN'] = HF_TOKEN
api = HfApi(token=HF_TOKEN)
identity = api.whoami()
HF_REPO_ID = f"{identity['name']}/coffee-sni21-vadcp-pilot"
api.create_repo(repo_id=HF_REPO_ID, repo_type='dataset', private=True, exist_ok=True)
preflight_path = Path('/content/hf_preflight.txt')
preflight_path.write_text('SNI-21 pilot backup preflight\n', encoding='utf-8')
api.upload_file(
    path_or_fileobj=str(preflight_path),
    path_in_repo='sni21-vadcp-pilot-seed42/preflight.txt',
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    commit_message='verify SNI-21 pilot artifact backup',
)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
EVIDENCE_ROOT.mkdir(parents=True, exist_ok=True)
print('DATA INPUT SIAP')
print('HASIL DRIVE :', RESULT_ROOT)
print('BACKUP HF   :', HF_REPO_ID)

In [ ]:
import shutil

print('MULAI PILOT 200 SCENE/ARM — BUKAN TRAINING', flush=True)
pilot = run_sni21_colab_setup(
    ADRIAN_ARCHIVE,
    FARUQ_ARCHIVE,
    CROP_ROOT,
    '/content',
    profile='pilot',
    seed=42,
)
assert pilot['training_ready'], pilot
assert pilot['training_executed'] is False
assert pilot['test_accessed'] is False

evidence_files = [
    Path('/content/sni21-colab-pilot-summary.json'),
    Path(pilot['setup_summary']),
]
for arm in ('A1', 'A2'):
    evidence_files.extend([
        Path(pilot['arms'][arm]['audit']),
        Path(pilot['arms'][arm]['raw_contact_sheet']),
        Path(pilot['arms'][arm]['overlay_contact_sheet']),
    ])
for source in evidence_files:
    assert source.is_file(), source
    target = EVIDENCE_ROOT / source.name
    shutil.copy2(source, target)
    print('EVIDENCE SAVED:', target)
print('PILOT TRAINING_READY = True. TEST TIDAK DIAKSES.')

In [ ]:
from IPython.display import Image as DisplayImage, display

for arm in ('A1', 'A2'):
    print('\n' + arm + ' RAW')
    display(DisplayImage(filename=pilot['arms'][arm]['raw_contact_sheet'], width=900))
    print(arm + ' OVERLAY')
    display(DisplayImage(filename=pilot['arms'][arm]['overlay_contact_sheet'], width=900))
print('Periksa gambar di atas. Cell berikut tidak berjalan otomatis.')

## Training screening

Jika contact sheet wajar, ubah `RUN_TRAINING = False` menjadi `True`, lalu jalankan cell ini. Hanya validation yang dievaluasi. Checkpoint ditulis ke Drive dan disinkronkan ke Hugging Face setiap kali model menyimpan checkpoint.

In [ ]:
import json
from pathlib import Path

RUN_TRAINING = False

if not RUN_TRAINING:
    print('TRAINING BELUM DIJALANKAN. Ubah RUN_TRAINING menjadi True setelah visual pilot disetujui.')
else:
    if 'pilot' not in globals():
        pilot_summary_path = Path('/content/sni21-colab-pilot-summary.json')
        if not pilot_summary_path.is_file():
            raise RuntimeError(
                'State pilot tidak ditemukan. Runtime kemungkinan telah reset; '
                'jalankan kembali cell setup pilot sebelum training.'
            )
        pilot = json.loads(pilot_summary_path.read_text(encoding='utf-8'))
        print('STATE PILOT DIPULIHKAN:', pilot_summary_path)
    assert pilot['training_ready'] is True
    assert torch.cuda.is_available()
    A0_ROOT = Path(pilot['a0_root'])
    A1_ROOT = Path(pilot['arms']['A1']['root'])
    A2_ROOT = Path(pilot['arms']['A2']['root'])
    for root in (A0_ROOT, A1_ROOT, A2_ROOT):
        assert (root / 'data.yaml').is_file(), root

    print('MULAI SCREENING A0/A1/A2 — seed 42, validation only', flush=True)
    result = run_vadcp_ablation(
        {'A0': A0_ROOT, 'A1': A1_ROOT, 'A2': A2_ROOT},
        RESULT_ROOT,
        configs={
            'A0': REPO / 'configs/A0_yolo26n_screen.yaml',
            'A1': REPO / 'configs/A1_yolo26n_screen.yaml',
            'A2': REPO / 'configs/A2_yolo26n_screen.yaml',
        },
        seeds=(42,),
        device='0',
        resume=True,
        count_audit=True,
        verified_audits={
            'A0': A0_ROOT / 'post_materialization_audit.json',
            'A1': Path(pilot['arms']['A1']['audit']),
            'A2': Path(pilot['arms']['A2']['audit']),
        },
        hf_repo_id=HF_REPO_ID,
        hf_path_prefix='sni21-vadcp-pilot-seed42',
        hf_private=True,
        evaluation_split='val',
        open_test=False,
    )
    assert result['evaluation_split'] == 'val'
    assert result['test_opened'] is False
    print('SCREENING SELESAI:', result['summary'])

In [ ]:
import json

SUMMARY_PATH = RESULT_ROOT / 'reports/vadcp_ablation_summary.json'
if not SUMMARY_PATH.is_file():
    print('SUMMARY BELUM ADA — training belum selesai atau belum dijalankan.')
else:
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    assert summary['evaluation_split'] == 'val'
    assert summary['test_opened'] is False
    print('=== HASIL VALIDATION SEED 42 ===')
    for code, metrics in summary['aggregate'].items():
        macro = metrics.get('mAP50-95', {}).get('mean')
        recall = metrics.get('recall', {}).get('mean')
        worst = metrics.get('worst_class_map50_95', {}).get('mean')
        print(f'{code}: mAP50-95={macro:.4f} recall={recall:.4f} worst={worst:.4f}')
    print('DELTA:', json.dumps(summary['comparisons'], indent=2))
    print('TEST TETAP TERKUNCI.')